In [ ]:
# https://open-meteo.com/en/docs/historical-forecast-api?utm_source=chatgpt.com&latitude=55.7558&longitude=37.6173&start_date=2021-01-01&end_date=2026-09-18&hourly=temperature_2m,relative_humidity_2m,precipitation_probability,precipitation,rain,snowfall,snow_depth,weather_code,cloud_cover,pressure_msl#hourly_weather_variables
# Сайт с погодой

In [2]:
import polars as pl

# Делаем бустинг. 
# Если поле - float делаем отдельную value колонку
# Если поле - категория из топ 100 (одна из 31), то добавляем как отдельный категориальный признак под неё
# Делаем статистики временные по float и по категориальным полям
# Делаем много статистик, потом режем дерево через shap и формируем итоговую модель
# Статистики для категорий - counts в разрезе 4-3-2-1-0.5 дней (так я это вижу)
# Текущий час нужно добавить 
# Номер месяца
# День недели
# все категории из df_sensors + тег_инженерной_системы тэг первый  (split('-')[0]))

# Таргет - часов до события критического (обучаем модель "выживаемости")

# То как на инференсе обновлять прогнозы все эти - большой вопрос

pl.Config.set_tbl_rows(10)


# НУЖНО СДЕЛАТЬ ТАК, ЧТОБЫ 
# 2025 лучше поменять на всякий случай 
lf = (
    pl.concat([
        pl.scan_parquet(f"data/ext-journal-{year}.parquet")
        for year in [2024, 2025, 2026]
        ])
    # .filter(pl.col("дата") != "дата")
    .cast({"дата": pl.Date})
)
df_sensors = pl.scan_parquet('data/справочник_каналов_датчиков.parquet')
df_territory = pl.scan_parquet('data/справочник_объектов_диспетчер.parquet')

In [3]:
DF_SENSORS: pl.LazyFrame = (
    pl.scan_parquet('data/справочник_каналов_датчиков.parquet')
    .select(
        pl.col('ид_канала_данных'),
        pl.col('тип_инж_системы'),
        pl.col('тип_датчика'),
        pl.col('ид_объект').cast(pl.Int64),
        pl.col('тег_инженерной_системы').str.split('-').list.get(0)
    )
)
DF_TERRITORY: pl.LazyFrame = (
    pl.scan_parquet('data/справочник_объектов_диспетчер.parquet')
    .drop('диспетчерское_название_объекта')
)
EXPECTED_WEATHER: pl.LazyFrame = (
    pl.scan_csv('data/open-meteo-55.75N37.63E140m.csv')
    .with_columns(
        pl.col("time")
        .str.to_datetime()
        + pl.duration(hours=3)
    )
)
MIN_CAT_VALUE_COUNT: int = 100


def preproc(
        lf: pl.LazyFrame
    ) -> pl.LazyFrame:
    '''Добавляет новые фичи в данные'''

    # Приводим к нужному формату типы
    lf = (
        lf
        .with_columns(
            pl.col("ид_события").cast(pl.Int64),
            pl.col("ид_канала_данных").cast(pl.Int64),
            pl.col("дата").cast(pl.Date),
            pl.col("время").str.to_time("%H:%M:%S"),
            pl.col("тревожное").eq("t").cast(pl.Int8),
            pl.col("значение_датчика").cast(pl.Float64, strict=False).alias("value_float"),
    
            pl.when(
                pl.col("значение_датчика").cast(pl.Float64, strict=False).is_null()
            )
            .then(pl.col("значение_датчика"))
            .otherwise(None)
            .alias("value_cat"),
        )
        .drop('значение_датчика')
    )

    # Делаем из значение_датчика категориальных - отдельные категории 
    cat_values = (
        lf
        .drop_nulls('value_cat')
        .group_by("value_cat")
        .agg(pl.len().alias("count"))
        .filter(pl.col("count") >= MIN_CAT_VALUE_COUNT)
        .select('value_cat')
        .collect()
        ['value_cat']
        .to_list()
    )

    lf = (
        lf
        .with_columns(
            pl.when(pl.col("value_cat") == cat)
            .then(1)
            .otherwise(0)
            .cast(pl.Int8)
            .alias(f"value_cat__{cat}")
            for cat in cat_values
        )
        .drop('value_cat')
    )

    # категории из DF_SENSORS и DF_TERRITORY
    lf = (
        lf
        .join(
            DF_SENSORS
            .join(
                DF_TERRITORY,
                on='ид_объект',
                how='left'
            )
            .drop('ид_объект')
            .cast({'ид_канала_данных': pl.Int64}), 
            on='ид_канала_данных', 
            how='left'
        )
    )

    # Дата и время
    lf = lf.with_columns(
        pl.col("дата").dt.weekday().alias("day_of_week"),
        pl.col("дата").dt.month().alias("month"),
        pl.col("время").dt.hour().alias("hour"),
    )

    # Погода
    lf = (
        lf
        .with_columns(
            pl.datetime(
                pl.col("дата").dt.year(),
                pl.col("дата").dt.month(),
                pl.col("дата").dt.day(),
                pl.col("время").dt.hour(),
                0,
                0,
            ).alias("datetime")
        )
        .join(
            EXPECTED_WEATHER,
            left_on="datetime",
            right_on="time",
            how="left",
        )
    )

    return lf

In [4]:
from datetime import timedelta

TARGET: str = "тревожное"
ID_COLS: list[str] = ["ид_канала_данных"]
TIME_COLS: list[str] = ["дата", "время"]
TIME_FIXED_CATS: list[str] = [
    "тип_инж_системы",
    "тип_датчика",
    "тег_инженерной_системы",
    "day_of_week",
    # "month", # Не на нескольких годах обучаюсь, поэтому убрать
    "hour",
    'иерархия_уровень',
    'родитель',
    'вид_объекта'
]
TIME_DYNAMIC_CATS: list[str] = [
    "value_cat__Есть питание",
    "value_cat__На охране",
    "value_cat__Дыма нет",
    "value_cat__Неисправен",
    "value_cat__Движения нет",
    "value_cat__Рычаг сдернут",
    "value_cat__Не определено",
    "value_cat__Температура ниже 3ºC",
    "value_cat__Вызов",
    "value_cat__Много неисправных устройств",
    "value_cat__Норма",
    "value_cat__Устройства на объекте исправны",
    "value_cat__Обнаружен газ",
    "value_cat__Выключен",
    "value_cat__Рычаг норма",
    "value_cat__01.01.1970 03:00:00",
    "value_cat__Обнаружено движение",
    "value_cat__Питание от батарей",
    "value_cat__Обнаружен дым",
    "value_cat__Работают все насосы в АНС",
    "value_cat__Включен",
    "value_cat__Разговор",
    "value_cat__В норме от +3 до +40",
    "value_cat__01.01.1970 03:00:01",
    "value_cat__Снято с охраны",
    "value_cat__Неопределен",
    "value_cat__Питание от сети",
    "value_cat__Отключено устройство",
    "value_cat__Обесточен",
    "value_cat__Затоплен",
    "value_cat__Не замкнут",
]
TIME_CONTINUOUS: list[str] = ["value_float"]
VALUE_STATISTICS_OVER_DAYS: list[float] = [
    3,
    2,
    1,
    12/24,
    6/24,
    4/24,
    2/24,
    1/24,
    0,
]
FUTURE_VALUES: list[str] = [
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'precipitation_probability (%)',
    'precipitation (mm)',
    'rain (mm)',
    'snowfall (cm)',
    'snow_depth (m)',
    'weather_code (wmo code)',
    'cloud_cover (%)',
    'pressure_msl (hPa)'
]
FUTUTE_LAGS_DAYS = [0, 1/24, 4/24, 8/24, 12/24, 16/24, 20/24, 24/24]


def _window_name(days: float) -> str:
    if days == 0:
        return "current"

    hours = days * 24

    if float(days).is_integer():
        return f"{int(days)}d"
    if float(hours).is_integer():
        return f"{int(hours)}h"

    minutes = hours * 60
    if float(minutes).is_integer():
        return f"{int(minutes)}m"

    return f"{days:g}d"

def create_train_dataset(lf: pl.LazyFrame) -> pl.LazyFrame:
    """Создаёт временные фичи, погодные фичи будущих часов и таргет."""

    date_col, time_col = TIME_COLS

    # Делаем единый datetime и сортируем данные по времени внутри каждого канала
    lf = (
        lf
        .with_columns(
            pl.datetime(
                pl.col(date_col).dt.year(),
                pl.col(date_col).dt.month(),
                pl.col(date_col).dt.day(),
                pl.col(time_col).dt.hour(),
                pl.col(time_col).dt.minute(),
                pl.col(time_col).dt.second(),
            ).alias("__datetime")
        )
        .sort([*ID_COLS, "__datetime"])
    )

    # Находим время предыдущего активного TARGET, не включая текущую строку
    prev_target_datetime = (
        pl.when(pl.col(TARGET) == 1)
        .then(pl.col("__datetime"))
        .otherwise(None)
        .shift(1)
        .fill_null(strategy="forward")
        .over(ID_COLS)
    )

    # Находим время следующего активного TARGET, не включая текущую строку
    next_target_datetime = (
        pl.when(pl.col(TARGET) == 1)
        .then(pl.col("__datetime"))
        .otherwise(None)
        .shift(-1)
        .fill_null(strategy="backward")
        .over(ID_COLS)
    )

    # Добавляем расстояние до предыдущего и следующего активного TARGET в часах
    lf = (
        lf
        .with_columns(
            prev_target_datetime.alias("__prev_target_datetime"),
            next_target_datetime.alias("__next_target_datetime"),
        )
        .with_columns(
            (
                (
                    pl.col("__datetime")
                    - pl.col("__prev_target_datetime")
                )
                .dt.total_seconds()
                / 3600.0
            ).alias("hours_since_prev_target"),

            (
                (
                    pl.col("__next_target_datetime")
                    - pl.col("__datetime")
                )
                .dt.total_seconds()
                / 3600.0
            ).alias(TARGET),
        )
    )

    feature_exprs: list[pl.Expr] = []

    # Строим временные статистики по каждому заданному окну
    for days in VALUE_STATISTICS_OVER_DAYS:
        suffix = _window_name(days)

        # Для текущей точки просто сохраняем текущее значение
        if days == 0:
            feature_exprs.extend(
                pl.col(col)
                .alias(f"{col}__sum_{suffix}")
                for col in TIME_DYNAMIC_CATS
            )

            feature_exprs.extend(
                pl.col(col)
                .alias(f"{col}__mean_{suffix}")
                for col in TIME_CONTINUOUS
            )

            continue

        window = timedelta(days=days)

        # Считаем количество категориальных событий за окно
        feature_exprs.extend(
            pl.col(col)
            .rolling_sum_by(
                by="__datetime",
                window_size=window,
                closed="right",
            )
            .over(ID_COLS)
            .alias(f"{col}__sum_{suffix}")
            for col in TIME_DYNAMIC_CATS
        )

        # Считаем среднее непрерывных значений за окно
        feature_exprs.extend(
            pl.col(col)
            .rolling_mean_by(
                by="__datetime",
                window_size=window,
                closed="right",
            )
            .over(ID_COLS)
            .alias(f"{col}__mean_{suffix}")
            for col in TIME_CONTINUOUS
        )

    # Добавляем rolling-фичи
    lf = lf.with_columns(feature_exprs)

    # ============================================================
    # Погодные прогнозы на текущий час и будущие часы
    # ============================================================

    for days in FUTUTE_LAGS_DAYS:
        hours = int(round(days * 24))
        suffix = _window_name(days)

        join_col = f"__future_datetime_{suffix}"

        # Для события 14:37:
        # 0h  -> 14:00
        # 1h  -> 15:00
        # 4h  -> 18:00
        # ...
        lf = lf.with_columns(
            (
                pl.col("__datetime").dt.truncate("1h")
                + pl.duration(hours=hours)
            ).alias(join_col)
        )

        weather_lf = (
            EXPECTED_WEATHER
            .select(
                pl.col("time"),
                *[
                    pl.col(col).alias(f"{col}__future_{suffix}")
                    for col in FUTURE_VALUES
                ],
            )
        )

        lf = (
            lf
            .join(
                weather_lf,
                left_on=join_col,
                right_on="time",
                how="left",
            )
            .drop(join_col)
        )

    # Удаляем технические и исходные колонки
    return (
        lf
        .drop(
            "__prev_target_datetime",
            "__next_target_datetime",
            "__datetime",
            *TIME_DYNAMIC_CATS,
            *TIME_CONTINUOUS,
            *FUTURE_VALUES,
            # *TIME_COLS,
            # *ID_COLS,
            'month'
        )
    )

In [5]:
from pathlib import Path
from tqdm.auto import tqdm
import polars as pl


def process_dataset_by_chunks(
    lf: pl.LazyFrame,
    output_dir: str = "data/preprocessed_dataset",
    chunk_size: int = 500,
) -> None:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Собираем в память только уникальные ID
    ids = (
        lf
        .select(ID_COLS)
        .unique()
        .collect()
    )

    n_chunks = (len(ids) + chunk_size - 1) // chunk_size

    for chunk_idx in tqdm(
        range(n_chunks),
        desc="Processing chunks",
        unit="chunk",
    ):
        start = chunk_idx * chunk_size

        chunk_ids = ids.slice(start, chunk_size)

        result_lf = (
            lf
            .join(
                chunk_ids.lazy(),
                on=ID_COLS,
                how="semi",
            )
            .pipe(create_train_dataset)
        )

        result_lf.sink_parquet(
            output_dir / f"chunk_{chunk_idx:05d}.parquet"
        )

/home/arseniy/miniconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import polars as plА 

preproc(lf).sink_parquet('промежуточный df.parquet')
df = pl.scan_parquet('промежуточный df.parquet')

process_dataset_by_chunks(df, chunk_size=100)

Processing chunks: 100%|██████████| 116/116 [19:16<00:00,  9.97s/chunk]
